In [1]:
!pip install pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 36.1 MB/s eta 0:00:00


In [2]:
import fitz  # PyMuPDF
import re

def extract_text_from_pdf(pdf_path):
    """
    Opens a PDF file and extracts all text page by page.
    """
    document = fitz.open(pdf_path)
    full_text = ""

    for page_num in range(len(document)):
        page = document.load_page(page_num)
        # Extract text and add a space to prevent words from merging across pages
        full_text += page.get_text("text") + " \n"

    return full_text

def chunk_contract_text(raw_text, min_words=10):
    """
    Splits the raw text into paragraph chunks and removes formatting artifacts.
    Filters out chunks that are too short to be meaningful clauses.
    """
    # Replace multiple spaces or tabs with a single space
    cleaned_text = re.sub(r'[ \t]+', ' ', raw_text)

    # Split the text by double newlines, which typically indicate a new paragraph or clause
    raw_chunks = re.split(r'\n\s*\n', cleaned_text)

    final_chunks = []

    for chunk in raw_chunks:
        # Remove single newlines within a paragraph to make it one continuous string
        chunk = chunk.replace('\n', ' ').strip()

        # Count words in the chunk
        word_count = len(chunk.split())

        # Keep the chunk only if it contains enough words to be a valid clause
        if word_count >= min_words:
            final_chunks.append(chunk)

    return final_chunks

def process_contract(pdf_path):
    """
    Master function to handle extraction and chunking.
    """
    print(f"Processing document: {pdf_path}")
    raw_text = extract_text_from_pdf(pdf_path)
    clauses = chunk_contract_text(raw_text)

    print(f"Extracted {len(clauses)} valid clauses/paragraphs.")
    return clauses

# --- Example Usage ---
# Upload a sample PDF contract to your Colab session, then run:
# my_clauses = process_contract('/content/sample_contract.pdf')
#
# To view the first 3 chunks:
# for i, clause in enumerate(my_clauses[:3]):
#     print(f"Chunk {i+1}: {clause}\n")

In [3]:
!pip install transformers torch

In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Define the exact Hugging Face model path
MODEL_NAME = "nlpaueb/legal-bert-base-uncased"

# Set up the device to use the GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Hardware assigned: {device}")

print("Downloading and loading the LegalBERT Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Downloading and loading the LegalBERT Model...")
# We use num_labels=2 because we are doing binary classification (Safe vs. Scam)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

# Move the heavy model to the GPU for faster processing
model.to(device)

print("LegalBERT architecture successfully loaded into memory.")

Hardware assigned: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: nlpaueb/legal-bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were new

LegalBERT architecture successfully loaded into memory.


In [5]:
import pandas as pd

# A small synthetic dataset of Safe (0) and Scam (1) clauses
data = {
    "clause_text": [
        # SAFE CLAUSES (Class 0)
        "The Employee's base salary shall be $75,000 per annum, paid bi-weekly in accordance with the Company's standard payroll practices.",
        "Either party may terminate this Agreement at any time, with or without cause, by providing two weeks' written notice.",
        "During the term of employment, the Employee will be eligible to participate in the Company's standard medical and dental insurance plans.",
        "The Employee agrees to maintain the confidentiality of all proprietary information and trade secrets belonging to the Employer.",
        "This Agreement shall be governed by and construed in accordance with the laws of the State of Delaware.",

        # SCAM / PREDATORY CLAUSES (Class 1)
        "Prior to the commencement date, the Employee must wire a non-refundable deposit of $850 to the Company's authorized vendor for the purchase of the home office workstation.",
        "The Employee agrees that the Company will send a cashier's check to cover office supplies. The Employee must deposit this check and immediately transfer the excess funds via Zelle to the IT department.",
        "Should the Employee resign within the first 12 months, they shall be liable for a $15,000 training reimbursement fee payable immediately upon termination.",
        "The candidate must provide their full bank routing number, account number, and online banking password to the HR portal for identity verification before an offer letter is finalized.",
        "Employee assumes all financial liability for errors made during data entry and agrees that the Company may deduct up to 100% of their wages to cover any perceived business losses."
    ],
    "is_scam": [0, 0, 0, 0, 0, 1, 1, 1, 1, 1]
}

df_contracts = pd.DataFrame(data)
print(f"Created a starter dataset with {df_contracts.shape[0]} clauses.")
print(df_contracts.head(10))

Created a starter dataset with 10 clauses.
                                         clause_text  is_scam
0  The Employee's base salary shall be $75,000 pe...        0
1  Either party may terminate this Agreement at a...        0
2  During the term of employment, the Employee wi...        0
3  The Employee agrees to maintain the confidenti...        0
4  This Agreement shall be governed by and constr...        0
5  Prior to the commencement date, the Employee m...        1
6  The Employee agrees that the Company will send...        1
7  Should the Employee resign within the first 12...        1
8  The candidate must provide their full bank rou...        1
9  Employee assumes all financial liability for e...        1


In [6]:
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from tqdm import tqdm

# 1. Create a custom PyTorch Dataset
class ContractDataset(Dataset):
    """
    This class handles the translation from raw English text into
    the numerical tokens that LegalBERT requires.
    """
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]

        # Tokenize the text: pad short sentences, truncate long ones
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

# 2. Prepare the DataLoader
# We feed our small pandas DataFrame into the custom dataset class
dataset = ContractDataset(
    texts=df_contracts['clause_text'].tolist(),
    labels=df_contracts['is_scam'].tolist(),
    tokenizer=tokenizer
)

# DataLoader feeds the data to the GPU in small chunks (batches)
# We use batch_size=2 because our sample dataset only has 10 rows
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

# 3. Setup the Optimizer
# We are now using PyTorch's native AdamW optimizer
optimizer = AdamW(model.parameters(), lr=2e-5)

# 4. The Training Loop
epochs = 3

print("Starting the training process...")
model.train() # Put the model into training mode

for epoch in range(epochs):
    total_loss = 0

    # tqdm creates a progress bar for the terminal
    for batch in tqdm(dataloader, desc=f"Epoch {epoch + 1}/{epochs}"):

        # Move batch data to the GPU
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        # Clear old calculations
        optimizer.zero_grad()

        # Forward pass: the model guesses if the clauses are safe or scams
        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)

        # How wrong was the model? (Calculate Loss)
        loss = outputs.loss
        total_loss += loss.item()

        # Backward pass: calculate how to adjust the weights to be less wrong
        loss.backward()

        # Update the model's brain
        optimizer.step()

    avg_loss = total_loss / len(dataloader)
    print(f"Average Loss for Epoch {epoch + 1}: {avg_loss:.4f}\n")

print("Training complete. LegalBERT is now tuned to your dataset.")

Starting the training process...


Epoch 1/3:   0%|          | 0/5 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Epoch 1/3: 100%|██████████| 5/5 [00:02<00:00,  2.30it/s]


Average Loss for Epoch 1: 0.7484



Epoch 2/3: 100%|██████████| 5/5 [00:01<00:00,  3.64it/s]


Average Loss for Epoch 2: 0.6372



Epoch 3/3: 100%|██████████| 5/5 [00:01<00:00,  3.44it/s]

Average Loss for Epoch 3: 0.5837

Training complete. LegalBERT is now tuned to your dataset.


In [7]:
import torch
import torch.nn.functional as F

def analyze_clause(text, threshold=0.50):
    """
    Takes a single clause, tokenizes it, and asks the trained LegalBERT
    model to predict if it is predatory.
    """
    # 1. Put the model in evaluation mode (turns off training mechanics)
    model.eval()

    # 2. Tokenize the input text exactly like we did during training
    encoding = tokenizer(
        text,
        add_special_tokens=True,
        max_length=128,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )

    # 3. Move the data to the GPU
    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)

    # 4. Run the prediction without calculating gradients (saves memory/time)
    with torch.no_grad():
        outputs = model(input_ids, attention_mask=attention_mask)
        logits = outputs.logits

        # Convert raw model outputs (logits) into standard percentages (0.0 to 1.0)
        probabilities = F.softmax(logits, dim=1)

        # Grab the probability specifically for Class 1 (Scam)
        scam_prob = probabilities[0][1].item()

    is_scam = scam_prob >= threshold
    return is_scam, scam_prob

# ==========================================
# TESTING THE INFERENCE ENGINE
# ==========================================

# Let's test it on two brand new sentences the model has never seen
test_clauses = [
    "The Employer will provide a company laptop and cover all standard software licensing fees required for the role.",
    "The new hire is required to send $1,200 via cryptocurrency to the onboarding manager for specialized equipment processing."
]

print("Scanning clauses...\n")

for i, clause in enumerate(test_clauses):
    is_scam, prob = analyze_clause(clause)

    print("-" * 70)
    print(f"Clause {i+1}: {clause}")
    if is_scam:
        print(f"Verdict: PREDATORY CLAUSE DETECTED (Confidence: {prob:.1%})")
    else:
        print(f"Verdict: Standard Term (Scam Probability: {prob:.1%})")

print("-" * 70)

Scanning clauses...

----------------------------------------------------------------------
Clause 1: The Employer will provide a company laptop and cover all standard software licensing fees required for the role.
Verdict: Standard Term (Scam Probability: 48.0%)
----------------------------------------------------------------------
Clause 2: The new hire is required to send $1,200 via cryptocurrency to the onboarding manager for specialized equipment processing.
Verdict: PREDATORY CLAUSE DETECTED (Confidence: 53.8%)
----------------------------------------------------------------------


In [15]:
import fitz  # PyMuPDF
import re

def generate_contract_risk_report(pdf_path, threshold=0.50):
    """
    Reads a PDF contract, breaks it into clauses, analyzes each one
    using the fine-tuned LegalBERT model, and prints a risk report.
    """
    print("=" * 70)
    print(f"CONTRACT RISK ASSESSMENT REPORT")
    print(f"Document: {pdf_path}")
    print("=" * 70)

    # 1. Extract text from PDF
    try:
        document = fitz.open(pdf_path)
    except Exception as e:
        print(f"Error opening PDF: {e}")
        return

    full_text = ""
    for page_num in range(len(document)):
        page = document.load_page(page_num)
        full_text += page.get_text("text") + " \n"

    # 2. Chunk text into logical clauses
    cleaned_text = re.sub(r'[ \t]+', ' ', full_text)
    raw_chunks = re.split(r'\n\s*\n', cleaned_text)

    clauses = []
    for chunk in raw_chunks:
        chunk = chunk.replace('\n', ' ').strip()
        if len(chunk.split()) >= 10:  # Ignore headers/footers
            clauses.append(chunk)

    print(f"Scanning {len(clauses)} valid clauses...\n")

    # 3. Analyze each clause using our inference engine
    flagged_clauses = []

    for i, clause in enumerate(clauses):
        # We use the analyze_clause function we defined previously
        is_scam, prob = analyze_clause(clause, threshold=threshold)

        if is_scam:
            flagged_clauses.append({
                "clause_num": i + 1,
                "text": clause,
                "confidence": prob
            })

    # 4. Generate the final output report
    if len(flagged_clauses) == 0:
        print("FINAL VERDICT: Clean.")
        print("No predatory clauses detected. The contract appears standard.")
    else:
        print(f"FINAL VERDICT: HIGH RISK")
        print(f"Flagged {len(flagged_clauses)} potentially predatory clause(s) for manual review.\n")
        print("-" * 70)

        for item in flagged_clauses:
            print(f"Location: Clause {item['clause_num']}")
            print(f"Threat Confidence: {item['confidence']:.1%}")
            print(f"Text: {item['text']}\n")

    print("=" * 70)

# --- Example Usage ---
# If you have a PDF named 'sample_offer.pdf' uploaded to Colab, run:
# generate_contract_risk_report('/content/sample_offer.pdf')

In [16]:
generate_contract_risk_report('/content/Sample Contract Scam.pdf')

CONTRACT RISK ASSESSMENT REPORT
Document: /content/Sample Contract Scam.pdf
Scanning 2 valid clauses...

FINAL VERDICT: HIGH RISK
Flagged 2 potentially predatory clause(s) for manual review.

----------------------------------------------------------------------
Location: Clause 1
Threat Confidence: 52.7%
Text: Common Legal Terminology in Employment Contracts  Employment Agreement – A legally binding contract outlining the terms and conditions of  employment between the employer and employee.  Probationary Period – An initial trial period during which the employer evaluates the  employee’s performance before confirming permanent employment status.  Compensation and Remuneration – The total payment provided to an employee for services  rendered, including salary, wages, bonuses, and incentives.  Benefits and Entitlements – Non-wage compensation such as health insurance, retirement  plans, paid leave, and other employer-provided privileges.  Scope of Duties and Responsibilities – A claus

In [10]:
import os
from google.colab import drive

# 1. Mount Google Drive (if it is not already mounted in this session)
drive.mount('/content/drive')

# 2. Define the exact folder path where you want to save the model
save_directory = '/content/drive/MyDrive/Contract_Scanner_Model'

# 3. Create the folder if it does not exist yet
if not os.path.exists(save_directory):
    os.makedirs(save_directory)

print(f"Saving the fine-tuned LegalBERT model and tokenizer to: {save_directory}")

# 4. Save the model's learned weights and configuration
model.save_pretrained(save_directory)

# 5. Save the tokenizer
tokenizer.save_pretrained(save_directory)

print("Save complete. Your custom AI is safely backed up to Google Drive.")

Mounted at /content/drive
Saving the fine-tuned LegalBERT model and tokenizer to: /content/drive/MyDrive/Contract_Scanner_Model


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Save complete. Your custom AI is safely backed up to Google Drive.
